# BirdCLEF 2026 — Baseline Audio Classification Model

This notebook builds a simple baseline model for BirdCLEF 2026.

Goals:
- load a subset of labeled training recordings
- extract simple audio features
- train a baseline classifier
- evaluate performance on a validation split

This baseline will serve as a starting point for later improvements.

In [44]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [45]:
train = pd.read_csv("../train.csv")
taxonomy = pd.read_csv("../taxonomy.csv")

print("Train shape:", train.shape)
print("Taxonomy shape:", taxonomy.shape)

Train shape: (35549, 15)
Taxonomy shape: (234, 5)


In [46]:
top_labels = train["primary_label"].value_counts().head(10).index

subset = (
    train[train["primary_label"].isin(top_labels)]
    .dropna(subset=["primary_label", "filename", "collection"])
    .sample(800, random_state=42)
    .reset_index(drop=True)
)

subset["primary_label"].value_counts().head()

primary_label
banana     91
socfly1    88
yeofly1    88
fepowl     84
coffal1    83
Name: count, dtype: int64

## Feature Extraction Strategy

For the baseline model, we will extract a compact summary of each audio clip.

Rather than training directly on full spectrogram images, we begin with simple engineered features:
- MFCC means
- MFCC standard deviations
- zero crossing rate
- spectral centroid
- spectral bandwidth

These features provide a lightweight numerical summary of each recording and are commonly used in audio classification tasks.

In [47]:
def get_audio_path(row):
    filename = row["filename"]
    collection = row["collection"]

    option_1 = f"../train_audio/{filename}"
    option_2 = f"../train_audio/{collection}/{filename}"

    if os.path.exists(option_1):
        return option_1
    if os.path.exists(option_2):
        return option_2
    return None

In [48]:
def extract_features(file_path, sr=32000, duration=5):
    try:
        y, sr = librosa.load(file_path, sr=sr)

        features_list = []

        for i in range(0, len(y), sr * duration):
            segment = y[i:i + sr * duration]

            if len(segment) < sr * duration:
                continue

            mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=20)
            delta_mfcc = librosa.feature.delta(mfcc)
            delta2_mfcc = librosa.feature.delta(mfcc, order=2)

            zcr = librosa.feature.zero_crossing_rate(segment)
            spec_centroid = librosa.feature.spectral_centroid(y=segment, sr=sr)
            spec_bandwidth = librosa.feature.spectral_bandwidth(y=segment, sr=sr)

            features = np.concatenate([
                mfcc.mean(axis=1),
                mfcc.std(axis=1),
                delta_mfcc.mean(axis=1),
                delta_mfcc.std(axis=1),
                delta2_mfcc.mean(axis=1),
                delta2_mfcc.std(axis=1),
                [zcr.mean(), zcr.std()],
                [spec_centroid.mean(), spec_centroid.std()],
                [spec_bandwidth.mean(), spec_bandwidth.std()]
            ])

            features_list.append(features)

        return features_list

    except:
        return None

In [49]:
X = []
y = []

for _, row in subset.iterrows():
    path = get_audio_path(row)

    if path is None:
        continue

    feature_segments = extract_features(path)

    if feature_segments is None:
        continue

    for features in feature_segments:
        X.append(features)
        y.append(row["primary_label"])

X = np.array(X)
y = np.array(y)

In [55]:
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (4895, 126)
Target shape: (4895,)


In [56]:
print("Number of classes:", len(np.unique(y)))

Number of classes: 10


In [50]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Number of classes in subset:", len(label_encoder.classes_))

Number of classes in subset: 10


In [51]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [52]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [53]:
y_pred = model.predict(X_valid)

print("Validation accuracy:", accuracy_score(y_valid, y_pred))

Validation accuracy: 0.7824310520939735


In [54]:
print(classification_report(y_valid, y_pred, target_names=label_encoder.classes_))

              precision    recall  f1-score   support

      banana       0.87      0.84      0.86        96
     coffal1       0.66      0.91      0.76       161
      compau       0.79      0.91      0.84       118
      fepowl       0.87      0.60      0.71       100
      houspa       0.83      0.74      0.78        70
      osprey       0.72      0.64      0.68        66
     rubthr1       0.88      0.63      0.73        91
     socfly1       0.84      0.73      0.78        97
     soulap1       0.92      0.79      0.85        62
     yeofly1       0.75      0.85      0.80       118

    accuracy                           0.78       979
   macro avg       0.81      0.76      0.78       979
weighted avg       0.80      0.78      0.78       979



## Baseline Model Results

Using segmented audio (5-second windows) significantly improved performance:

- Feature samples increased from ~800 to ~4,800
- Validation accuracy improved from ~0.50 to ~0.78
- Model performance is now consistent across all classes

This demonstrates the importance of aligning data preprocessing with the natural structure of bird vocalizations.